In [1]:
import pandas as pd  # Import pandas for structured data analysis.
import numpy as np  # Import NumPy for numerical diagnostics.
from pathlib import Path  # Import Path for file-safe references.
from IPython.display import display  # Import display for readable audit tables.
from scipy import stats  # Import statistical tests for audit checks.
FILE_PATH = "raw_data/marketing_events.csv"  # Point to the project raw dataset.
audit_findings = []  # Create a register for evidence-based findings.
print("BUSINESS INSIGHT: Customer Retention and Churn")  # State the consulting context for the audit.
print("BUSINESS PROBLEM: Identify customer behaviours associated with inactivity and churn.")  # State the business problem being investigated.
print("AUDIT LENS: retention, repeat purchase, payment friction, service experience")  # State the signals relevant to this problem.


BUSINESS INSIGHT: Customer Retention and Churn
BUSINESS PROBLEM: Identify customer behaviours associated with inactivity and churn.
AUDIT LENS: retention, repeat purchase, payment friction, service experience


In [2]:
# Resolve the dataset relative to the current working directory or its parent folders.
requested_path = Path(FILE_PATH)
search_roots = [Path.cwd(), *Path.cwd().parents]
candidates = [root / requested_path for root in search_roots]

resolved_path = next((path for path in candidates if path.is_file()), None)
if resolved_path is None:
    raise FileNotFoundError(f"Could not find {FILE_PATH}. Searched: {candidates}")

FILE_PATH = str(resolved_path)
df = pd.read_csv(FILE_PATH)

print(f"Loaded {Path(FILE_PATH).name}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
display(df.head())  # Load the raw dataset before making changes.
print(f"Loaded {Path(FILE_PATH).name}")  # Confirm the source dataset loaded.
print(f"Rows: {len(df):,}")  # Show the available observation count.
print(f"Columns: {len(df.columns):,}")  # Show the available field count.
display(df.head())  # Inspect representative raw records.


Loaded marketing_events.csv
Rows: 30,000
Columns: 8


,event_id,customer_id,campaign_id,event_timestamp,event_type,device_type,channel,session_id
0,EVT-00000001,CUS-002490,CMP-0014,2025-09-01 19:06:00,Add To Cart,Tablet,Email,SES-627680706
1,EVT-00000002,CUS-009197,CMP-0040,2023-11-10 07:36:00,Impression,Desktop,Email,SES-290709858
2,EVT-00000003,CUS-008125,CMP-0009,2022-02-12 09:12:00,Impression,Desktop,Push,SES-349101587
3,EVT-00000004,CUS-005282,CMP-0013,2026-02-04 11:46:00,Product View,Tablet,Push,SES-566163779
4,EVT-00000005,CUS-011417,CMP-0042,2022-06-04 13:25:00,Open,Mobile,Paid Search,SES-197036241


Loaded marketing_events.csv
Rows: 30,000
Columns: 8


,event_id,customer_id,campaign_id,event_timestamp,event_type,device_type,channel,session_id
0,EVT-00000001,CUS-002490,CMP-0014,2025-09-01 19:06:00,Add To Cart,Tablet,Email,SES-627680706
1,EVT-00000002,CUS-009197,CMP-0040,2023-11-10 07:36:00,Impression,Desktop,Email,SES-290709858
2,EVT-00000003,CUS-008125,CMP-0009,2022-02-12 09:12:00,Impression,Desktop,Push,SES-349101587
3,EVT-00000004,CUS-005282,CMP-0013,2026-02-04 11:46:00,Product View,Tablet,Push,SES-566163779
4,EVT-00000005,CUS-011417,CMP-0042,2022-06-04 13:25:00,Open,Mobile,Paid Search,SES-197036241


In [3]:
overview = pd.DataFrame({"metric":["rows","columns","duplicates","missing_cells"],"value":[len(df),len(df.columns),int(df.duplicated().sum()),int(df.isna().sum().sum())]})  # Build an initial data-quality summary.
display(overview)  # Review the initial quality position.
print("Decision point: determine which findings require remediation.")  # Make the audit decision explicit.


,metric,value
0,rows,30000
1,columns,8
2,duplicates,0
3,missing_cells,0


Decision point: determine which findings require remediation.


In [4]:
schema = pd.DataFrame({"column":df.columns,"dtype":df.dtypes.astype(str).values,"non_null":df.notna().sum().values,"missing":df.isna().sum().values,"unique":df.nunique(dropna=True).values})  # Profile schema completeness and cardinality.
display(schema)  # Inspect field-level structural evidence.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()  # Identify numeric fields for statistical checks.
text_cols = df.select_dtypes(include=["object","string"]).columns.tolist()  # Identify text fields for categorical checks.
date_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]  # Identify likely temporal fields.


,column,dtype,non_null,missing,unique
0,event_id,str,30000,0,30000
1,customer_id,str,30000,0,11026
2,campaign_id,str,30000,0,50
3,event_timestamp,str,30000,0,29600
4,event_type,str,30000,0,6
5,device_type,str,30000,0,9
6,channel,str,30000,0,5
7,session_id,str,30000,0,29999


In [5]:
missing = df.isna().sum().sort_values(ascending=False)  # Measure explicit missingness by field.
missing = missing[missing.gt(0)]  # Keep only fields with missing values.
display(missing.to_frame("missing_count"))  # Inspect missing-value concentration.
for col in missing.index: audit_findings.append({"issue":"missing_values","column":col,"count":int(missing[col])})  # Register observed missing-value evidence.


,missing_count


In [6]:
duplicates = int(df.duplicated().sum())  # Measure exact duplicate records.
audit_findings.append({"issue":"duplicate_rows","count":duplicates})  # Register duplicate-row evidence.
print(f"Duplicate rows identified: {duplicates:,}")  # Report duplicate-row evidence.


Duplicate rows identified: 0


In [7]:
numeric_audit = df[numeric_cols].describe().T if numeric_cols else pd.DataFrame()  # Summarise numeric distributions.
display(numeric_audit)  # Inspect scale, spread and potential extremes.
if numeric_cols: outlier_rates = ((df[numeric_cols] < df[numeric_cols].quantile(.25) - 1.5*(df[numeric_cols].quantile(.75)-df[numeric_cols].quantile(.25))) | (df[numeric_cols] > df[numeric_cols].quantile(.75) + 1.5*(df[numeric_cols].quantile(.75)-df[numeric_cols].quantile(.25)))).mean().sort_values(ascending=False)  # Estimate IQR-based extreme-value rates.
if numeric_cols: display(outlier_rates.to_frame("iqr_extreme_rate"))  # Inspect fields requiring business review.


""


In [8]:
category_audit = []  # Create categorical consistency checks.
for col in text_cols: category_audit.append({"column":col,"unique":int(df[col].nunique(dropna=True)),"blank":int(df[col].astype("string").str.strip().eq("").sum()),"top_values":df[col].value_counts(dropna=False).head(5).to_dict()})  # Profile text fields for inconsistent values.
display(pd.DataFrame(category_audit))  # Review categorical concentration and blanks.


,column,unique,blank,top_values
0,event_id,30000,0,"{'EVT-00000001': 1, 'EVT-00000002': 1, 'EVT-00..."
1,customer_id,11026,0,"{'CUS-002351': 11, 'CUS-010122': 10, 'CUS-0116..."
2,campaign_id,50,0,"{'CMP-0018': 656, 'CMP-0041': 633, 'CMP-0011':..."
3,event_timestamp,29600,0,"{'2024-10-29 10:12:00': 3, '2023-11-09 19:21:0..."
4,event_type,6,0,"{'Impression': 10660, 'Product View': 6583, 'A..."
5,device_type,9,0,"{'Tablet': 10015, 'Desktop': 9996, 'Mobile': 9..."
6,channel,5,0,"{'SMS': 7066, 'Social': 6651, 'Paid Search': 6..."
7,session_id,29999,0,"{'SES-659405967': 2, 'SES-627680706': 1, 'SES-..."


In [9]:
date_audit = []  # Create temporal field diagnostics.
for col in date_cols: parsed = pd.to_datetime(df[col], errors="coerce"); date_audit.append({"column":col,"parse_failures":int(parsed.isna().sum()-df[col].isna().sum()),"min":parsed.min(),"max":parsed.max()})  # Test temporal fields for parseability and range.
display(pd.DataFrame(date_audit))  # Review date integrity before analysis.


,column,parse_failures,min,max
0,event_timestamp,0,2020-01-04 00:32:00,2026-07-24 23:45:00


In [10]:
identifier_audit = []  # Create identifier uniqueness diagnostics.
for col in df.columns: identifier_audit.append({"column":col,"unique_ratio":round(df[col].nunique(dropna=True)/max(len(df),1),3)})  # Measure field-level uniqueness.
identifier_audit = pd.DataFrame(identifier_audit).sort_values("unique_ratio",ascending=False)  # Rank potential identifiers and keys.
display(identifier_audit.head(15))  # Inspect candidate identifiers and high-cardinality fields.


,column,unique_ratio
0,event_id,1.000
7,session_id,1.000
3,event_timestamp,0.987
1,customer_id,0.368
2,campaign_id,0.002
4,event_type,0.000
5,device_type,0.000
6,channel,0.000


In [11]:
numeric_pairs = []  # Create relationship diagnostics for numeric fields.
if len(numeric_cols) > 1: numeric_pairs = df[numeric_cols].corr(numeric_only=True).stack().reset_index(name="correlation")  # Measure numeric relationships for diagnostic context.
if numeric_pairs != []: display(numeric_pairs.sort_values("correlation",key=lambda s:s.abs(),ascending=False).head(20))  # Inspect strongest observed numeric relationships.


In [12]:
finding_table = pd.DataFrame(audit_findings)  # Convert findings into a reviewable audit register.
if finding_table.empty: finding_table = pd.DataFrame([{ "issue":"none_detected_by_template", "count":0 }])  # Record when automated checks find no issues.
display(finding_table)  # Review the evidence before remediation.
print("Consulting decision: validate material findings against business rules before cleaning.")  # Prevent automatic treatment of every anomaly as an error.


,issue,count
0,duplicate_rows,0


Consulting decision: validate material findings against business rules before cleaning.


In [13]:
stem = Path(FILE_PATH).stem  # Capture the dataset stem for output naming.
audit_summary = pd.DataFrame({"dataset":[Path(FILE_PATH).name],"rows":[len(df)],"columns":[len(df.columns)],"duplicates":[duplicates],"missing_cells":[int(df.isna().sum().sum())]})  # Create an auditable executive summary.
display(audit_summary)  # Present the final audit snapshot.
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

audit_summary.to_csv(output_dir / f"{stem}_audit_summary.csv", index=False)


,dataset,rows,columns,duplicates,missing_cells
0,marketing_events.csv,30000,8,0,0
